# Single-Alpha QA-FedAvg Run (GPU-Friendly)

This notebook is meant for Kaggle or Colab GPU runtimes. It clones your `q-fedrag2` branch, installs the required dependencies without replacing the notebook's GPU-enabled PyTorch, runs exactly one alpha configuration of the mechanism-track `federated_noisy_qa.py`, streams live progress, and packages the output artifacts for download.

Use four notebook copies in parallel by changing only `ALPHA` in Cell 1. The rest of the configuration should stay identical across the four runs.

In [1]:
from pathlib import Path

REPO_URL = "https://github.com/Abishek-Chakravarthy/fed-rag.git"
REPO_BRANCH = "q-fedrag2"

ALPHA = 0.7
SEED = 42

# Mechanism benchmark defaults.
# Keep BENCHMARK_MODE fixed across the 4 alpha notebooks.
BENCHMARK_MODE = "mechanism"
NOISE_MODE = "shuffle"
NOISE_RATIO = 0.0
ROUNDS = 4
LOCAL_EPOCHS = 1

# Keep this notebook single-alpha. Run 4 notebook copies in parallel with:
# ALPHA = 0.0, 0.3, 0.7, 1.0
ALLOWED_ALPHAS = {0.0, 0.3, 0.7, 1.0}
if ALPHA not in ALLOWED_ALPHAS:
    raise ValueError(f"Use one of {sorted(ALLOWED_ALPHAS)} for the parallel alpha notebooks")

WORKSPACE = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content")
REPO_DIR = WORKSPACE / "fed-rag"
EXP_DIR = REPO_DIR / "zz_coderuns" / "quality_aware_fedrag" / "mechanism_benchmark"
EXPORT_DIR = WORKSPACE / f"qa_fedavg_mechanism_exports_{BENCHMARK_MODE}"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"WORKSPACE: {WORKSPACE}")
print(f"REPO_DIR : {REPO_DIR}")
print(f"EXP_DIR  : {EXP_DIR}")
print(f"TRACK    : {BENCHMARK_MODE}")
print(f"ALPHA    : {ALPHA}")
print(f"SEED     : {SEED}")

WORKSPACE: /kaggle/working
REPO_DIR : /kaggle/working/fed-rag
EXP_DIR  : /kaggle/working/fed-rag/zz_coderuns/quality_aware_fedrag/mechanism_benchmark
TRACK    : mechanism
ALPHA    : 0.7
SEED     : 42


In [2]:
import os
import shutil
import subprocess
import sys

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "--branch",
        REPO_BRANCH,
        "--single-branch",
        REPO_URL,
        str(REPO_DIR),
    ],
    check=True,
)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "protobuf>=4.25.3,<6",
    "accelerate",
    "datasets<3.0.0",
    "flwr==1.22.0",
    "pyarrow",
    "pydantic",
    "pydantic-settings",
    "transformers==4.48.0",
    "sentence-transformers==3.4.1",
    "peft",
    "matplotlib",
    "pandas",
    "tqdm",
], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR), "--no-deps"], check=True)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
print("Clone + install complete")


Cloning into '/kaggle/working/fed-rag'...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.5 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2024.6.1 which is incompatible.
black 26.3.1 requires pathspec>=1.0.0, but you have pathspec 0.12.1 which is incompatible.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompati

Clone + install complete


In [3]:
import importlib.metadata
import subprocess
import torch

print("torch version:", torch.__version__)
print("protobuf version:", importlib.metadata.version("protobuf"))
print("cuda available:", torch.cuda.is_available())
print("mps available :", torch.backends.mps.is_available())

if torch.cuda.is_available():
    print("gpu device:", torch.cuda.get_device_name(0))
    subprocess.run(["nvidia-smi"])
elif torch.backends.mps.is_available():
    print("Using Apple Metal backend")
else:
    print("WARNING: No GPU backend detected. Switch the notebook runtime to GPU.")


torch version: 2.10.0+cu128
cuda available: True
mps available : False
gpu device: Tesla T4
Thu Apr 23 13:36:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |     

In [4]:
def build_run_slug(alpha: float, seed: int, benchmark_mode: str, noise_mode: str, noise_ratio: float, rounds: int, local_epochs: int) -> str:
    track = benchmark_mode.replace("-", "_")
    mode = noise_mode.replace("-", "_")
    ratio = f"{noise_ratio:.2f}".replace(".", "p")
    return f"alpha_{alpha:.1f}_track_{track}_seed_{seed}_mode_{mode}_ratio_{ratio}_r{rounds}_e{local_epochs}"

RUN_SLUG = build_run_slug(ALPHA, SEED, BENCHMARK_MODE, NOISE_MODE, NOISE_RATIO, ROUNDS, LOCAL_EPOCHS)
print(RUN_SLUG)
print(f"Export root: {EXPORT_DIR}")

alpha_0.7_track_mechanism_seed_42_mode_shuffle_ratio_0p00_r4_e1
Export root: /kaggle/working/qa_fedavg_mechanism_exports_mechanism


In [5]:
import re
import subprocess
import sys
from tqdm.auto import tqdm

cmd = [
    sys.executable,
    "-u",
    "federated_noisy_qa.py",
    "--alpha", str(ALPHA),
    "--seed", str(SEED),
    "--benchmark-mode", BENCHMARK_MODE,
    "--noise-mode", NOISE_MODE,
    "--noise-ratio", str(NOISE_RATIO),
    "--rounds", str(ROUNDS),
    "--local-epochs", str(LOCAL_EPOCHS),
]

print("Running:", " ".join(cmd))
round_pattern = re.compile(r"Round\s+(\d+)\s+\|")
progress = tqdm(total=ROUNDS, desc=f"alpha={ALPHA:.1f}", unit="round")

process = subprocess.Popen(
    cmd,
    cwd=EXP_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")
    match = round_pattern.search(line)
    if match:
        progress.n = min(int(match.group(1)), ROUNDS)
        progress.refresh()

process.wait()
if process.returncode == 0:
    progress.n = ROUNDS
    progress.refresh()
progress.close()

if process.returncode != 0:
    raise subprocess.CalledProcessError(process.returncode, cmd)

Running: /usr/bin/python3 -u federated_noisy_qa.py --alpha 0.7 --seed 42 --benchmark-mode mechanism --noise-mode shuffle --noise-ratio 0.0 --rounds 4 --local-epochs 1


alpha=0.7:   0%|          | 0/4 [00:00<?, ?round/s]

2026-04-23 13:36:24.448480: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776951384.890511      85 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776951385.005540      85 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776951385.990756      85 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776951385.990815      85 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776951385.990818      85 computation_placer.cc:177] computation placer alr

CalledProcessError: Command '['/usr/bin/python3', '-u', 'federated_noisy_qa.py', '--alpha', '0.7', '--seed', '42', '--benchmark-mode', 'mechanism', '--noise-mode', 'shuffle', '--noise-ratio', '0.0', '--rounds', '4', '--local-epochs', '1']' returned non-zero exit status 1.

In [ ]:
import json
import pandas as pd
import shutil
import zipfile

csv_path = EXP_DIR / "output_csv_files" / f"results_{RUN_SLUG}.csv"
manifest_path = EXP_DIR / "output_csv_files" / f"manifest_{RUN_SLUG}.json"
acceptance_path = EXP_DIR / "output_csv_files" / f"acceptance_{RUN_SLUG}.json"
log_path = EXP_DIR / "output_log_files" / f"log_{RUN_SLUG}.log"

for path in [csv_path, manifest_path, acceptance_path, log_path]:
    print(path, path.exists())

run_export_dir = EXPORT_DIR / RUN_SLUG
run_export_dir.mkdir(parents=True, exist_ok=True)

for path in [csv_path, manifest_path, acceptance_path, log_path]:
    shutil.copy2(path, run_export_dir / path.name)

zip_path = EXPORT_DIR / f"{RUN_SLUG}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in run_export_dir.iterdir():
        zf.write(file_path, arcname=file_path.name)

print(f"Export folder: {run_export_dir}")
print(f"Zip file     : {zip_path}")

df = pd.read_csv(csv_path)
display(df)
print("Final row:")
display(df.tail(1))

with open(acceptance_path, "r", encoding="utf-8") as f:
    acceptance = json.load(f)
print("Acceptance report:")
print(json.dumps(acceptance, indent=2))

## What to download

Run four notebook copies in parallel with `ALPHA = 0.0`, `0.3`, `0.7`, and `1.0`.

For each notebook, download either:

- the `.zip` file shown above, or
- the whole `qa_fedavg_mechanism_exports_<track>/<run_slug>/` folder

Keep the same `BENCHMARK_MODE`, `SEED`, `ROUNDS`, and `LOCAL_EPOCHS` across all four notebooks.

After you collect all four alpha runs locally, use `compile_alpha_results.py` on the downloaded folders or CSV files.